# 05. Риски: моделирование Монте-Карло и дерево решений

**ЛР 4, шаг 2.** Ноутбук отвечает на вопрос спонсора «с какой вероятностью мы успеем в срок и уложимся в трудоёмкость» с учётом неопределённости оценок и рисков из [реестра рисков](../docs/risk-register.md).

Сценарии:

1. **Только неопределённость оценок** — без рисков.
2. **Оценки и риски до ответных мер** — риски наступают с вероятностями из реестра.
3. **После ответных мер** — вероятности снижены мерами из реестра.

Все прогоны используют фиксированное зерно, поэтому цифры воспроизводятся.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import date, timedelta

pd.set_option("display.float_format", lambda x: f"{x:,.2f}".replace(",", " "))
rng = np.random.default_rng(42)   # фиксированное зерно: у всех получатся одинаковые цифры

## 1. Исходные данные

Работы — те же, что в базовом плане ЛР 3 (`docs/schedule.md`): коды WBS, оценки O/M/P в часах, длительность в рабочих днях по плану и предшественники. Поле `gate` — день старта по календарному плану спринтов (Sprint 1 — дни 1–10, Sprint 2 — 11–20, Sprint 3 — 21–30), чтобы прогоны совпадали с базовым планом.

Риски взяты из реестра ЛР 4: вероятность и последствия до и после ответных мер.

In [ ]:
TASKS = [
    # code, wbs,  name,                                         O,  M,  P, days, gate, after
    ("A", "3.1", "Методика расчёта изохрон",                     3,  5,  8,  2,  0, ""),
    ("B", "3.2", "Изохроны школ",                                8, 12, 20,  4,  0, "A"),
    ("C", "3.3", "Изохроны поликлиник",                          8, 12, 20,  4,  0, "A"),
    ("D", "4.1", "Структура и макет карты",                      3,  5,  9,  2,  0, "A"),
    ("E", "5.1", "Методика сравнения районов",                   2,  3,  6,  1, 10, "B C"),
    ("F", "4.2", "Интерактивная карта",                          9, 14, 24,  5, 10, "B C D"),
    ("G", "4.3", "Обозначения и легенда",                        2,  4,  7,  2, 15, "F"),
    ("H", "5.2", "Сравнение доступности районов",                6,  9, 14,  3, 15, "E F"),
    ("I", "6.1", "Документация проекта",                         5,  8, 14,  3, 18, "G H"),
    ("J", "6.2", "Итоговый аналитический отчёт",                 4,  6, 10,  2, 21, "I"),
    ("K", "6.3", "Демонстрация и релиз",                         3,  4,  7,  2, 23, "J"),
    ("L", "6.4", "Приёмка и исправления",                        3,  4,  8,  2, 25, "K"),
]

RISKS = [
    # id,  описание,                                              task, p_before, days, hours, p_after, days_after, hours_after
    ("R1",  "Неполнота данных OSM: повторная проверка объектов",     "B",  0.50,    3,    6,     0.20,    3,          6),
    ("R2",  "Пешеходная сеть OSM непригодна для изохрон",            "F",  0.40,    3,   12,     0.10,    3,         12),
    ("R3",  "Сервис расчёта изохрон недоступен",                     "A",  0.20,    1,    4,     0.05,    1,          4),
    ("R4",  "Задержка критической работы выходит за буфер",          "H",  0.30,    3,    0,     0.30,    1,          0),
    ("R5",  "Данные устаревают к защите",                            "J",  0.15,    1,    2,     0.05,    1,          2),
    ("R7",  "Ошибки в параметрах расчёта изохрон",                   "C",  0.30,    2,    4,     0.10,    2,          4),
    ("R9",  "Частные клиники завышают доступность центра",           "C",  0.40,    1,    3,     0.15,    1,          3),
    ("R11", "Оценки изохрон и карты занижены",                       "F",  0.30,    3,    4,     0.15,    3,          4),
    ("R12", "Ограничения лицензий источников",                       "K",  0.10,    1,    2,     0.03,    1,          2),
]

START = date(2026, 11, 16)          # первый рабочий день плана
DEADLINE = date(2026, 12, 23)        # конец работ по базовому плану
BUFFER_END = date(2026, 12, 25)      # последний день буфера проекта
PLAN_HOURS = 91.1                    # ожидаемая трудоёмкость E из ЛР 3
RESERVE_HOURS = 15                   # резерв на риски из бюджета ЛР 3
N = 10_000                           # число прогонов

tasks = pd.DataFrame(TASKS, columns=["code", "wbs", "name", "O", "M", "P", "days", "gate", "after"]).set_index("code")
tasks["after"] = tasks["after"].str.split()
tasks["E"] = (tasks["O"] + 4 * tasks["M"] + tasks["P"]) / 6
risks = pd.DataFrame(RISKS, columns=["id", "name", "task", "p_before", "days_before", "hours_before",
                                     "p_after", "days_after", "hours_after"]).set_index("id")
risks["p_none"], risks["days_none"], risks["hours_none"] = 0.0, 0, 0
assert set(risks["task"]) <= set(tasks.index), "В рисках указана несуществующая работа"
tasks[["wbs", "name", "E", "days", "gate"]]

## 2. Календарь и порядок работ

In [ ]:
def workdays(start, n):
    days, d = [], start
    while len(days) < n:
        if d.weekday() < 5:
            days.append(d)
        d += timedelta(days=1)
    return days

CAL = workdays(START, 200)

def day_to_date(x):
    return CAL[max(int(np.ceil(x)) - 1, 0)]

def date_to_day(d):
    return CAL.index(d) + 1

succ = {k: [s for s in tasks.index if k in tasks.at[s, "after"]] for k in tasks.index}
order, seen = [], set()
def visit(k):
    if k in seen: return
    for p in tasks.at[k, "after"]: visit(p)
    seen.add(k); order.append(k)
for k in tasks.index: visit(k)

DEADLINE_DAY, BUFFER_DAY = date_to_day(DEADLINE), date_to_day(BUFFER_END)
print("Порядок работ:", " → ".join(order))
print(f"Срок по плану: день {DEADLINE_DAY} ({DEADLINE:%d.%m}), конец буфера: день {BUFFER_DAY} ({BUFFER_END:%d.%m})")

## 3. Модель

В каждом прогоне оценки O/M/P разыгрываются бета-распределением (PERT), длительности масштабируются пропорционально разыгранным часам, риски «срабатывают» независимо, после чего работы проходят прямой и обратный проход с учётом дат старта по спринтам.

In [ ]:
def beta_pert(o, m, p, size):
    if p == o:
        return np.full(size, float(m))
    a = 1 + 4 * (m - o) / (p - o)
    b = 1 + 4 * (p - m) / (p - o)
    return o + rng.beta(a, b, size) * (p - o)

def simulate(scenario):
    hours = pd.DataFrame({k: beta_pert(*tasks.loc[k, ["O", "M", "P"]], N) for k in tasks.index})
    dur = hours.div(tasks["E"]) * tasks["days"]
    fired = pd.DataFrame({r: rng.random(N) < risks.at[r, f"p_{scenario}"] for r in risks.index})
    for r in risks.index:
        t = risks.at[r, "task"]
        dur[t] += fired[r] * risks.at[r, f"days_{scenario}"]
        hours[t] += fired[r] * risks.at[r, f"hours_{scenario}"]
    ef = pd.DataFrame(index=dur.index)
    for k in order:
        preds = tasks.at[k, "after"]
        es = ef[preds].max(axis=1) if preds else 0
        es = np.maximum(es, tasks.at[k, "gate"])
        ef[k] = es + dur[k]
    finish = ef.max(axis=1)
    ls = pd.DataFrame(index=dur.index)
    for k in reversed(order):
        lf = ls[succ[k]].min(axis=1) if succ[k] else finish
        ls[k] = lf - dur[k]
    slack = ls[order] - (ef[order] - dur[order])
    critical = slack.abs() < 1e-6
    return dict(hours=hours, dur=dur, fired=fired, finish=finish, total_hours=hours.sum(axis=1), critical=critical)

only_estimates = simulate("none")
before = simulate("before")
after = simulate("after")
print("Готово:", N, "прогонов для каждого сценария")

## 4. Срок и трудоёмкость: вероятности

In [ ]:
def summary(res, title):
    f, h = res["finish"], res["total_hours"]
    p50, p85 = np.percentile(f, [50, 85])
    return {
        "Сценарий": title,
        "P(в срок), %": round(100 * (f <= DEADLINE_DAY).mean(), 1),
        "P(в буфер), %": round(100 * (f <= BUFFER_DAY).mean(), 1),
        "Срок P50": day_to_date(p50).strftime("%d.%m"),
        "Срок P85": day_to_date(p85).strftime("%d.%m"),
        "Часы P50": round(np.percentile(h, 50), 1),
        "Часы P85": round(np.percentile(h, 85), 1),
        f"P(часы <= {PLAN_HOURS:g}), %": round(100 * (h <= PLAN_HOURS).mean(), 1),
        f"P(часы <= {PLAN_HOURS + RESERVE_HOURS:g}), %": round(100 * (h <= PLAN_HOURS + RESERVE_HOURS).mean(), 1),
    }

result = pd.DataFrame([summary(only_estimates, "Только неопределённость оценок"),
                       summary(before, "Оценки и риски до ответных мер"),
                       summary(after, "Оценки и риски после ответных мер")]).set_index("Сценарий")
result.T

In [ ]:
from pathlib import Path
IMG = Path("../docs/img"); IMG.mkdir(parents=True, exist_ok=True)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
bins = np.arange(int(min(before["finish"].min(), after["finish"].min())), int(before["finish"].max()) + 2)
for res, label in [(before, "до ответных мер"), (after, "после ответных мер")]:
    axes[0].hist(res["finish"], bins=bins, alpha=0.5, label=label)
    axes[1].hist(res["total_hours"], bins=40, alpha=0.5, label=label)
axes[0].axvline(DEADLINE_DAY, color="black", ls="--", label=f"срок {DEADLINE:%d.%m}")
axes[0].axvline(BUFFER_DAY, color="red", ls="--", label=f"конец буфера {BUFFER_END:%d.%m}")
axes[0].set_xlabel("Окончание, рабочий день от старта"); axes[0].set_ylabel("Число прогонов"); axes[0].legend()
axes[1].axvline(PLAN_HOURS, color="black", ls="--", label=f"план {PLAN_HOURS:g} ч")
axes[1].axvline(PLAN_HOURS + RESERVE_HOURS, color="red", ls="--", label="план + резерв")
axes[1].set_xlabel("Трудоёмкость, ч"); axes[1].legend()
fig.suptitle("Монте-Карло: срок и трудоёмкость проекта")
fig.tight_layout()
fig.savefig(IMG / "monte_carlo.png", dpi=150)
print("Сохранено:", (IMG / "monte_carlo.png").resolve())

## 5. Какие работы и риски сильнее всего влияют на срок

In [ ]:
res = before
crit = pd.DataFrame({
    "wbs": tasks["wbs"], "name": tasks["name"],
    "Индекс критичности, %": (100 * res["critical"].mean()).round(1),
    "Чувствительность": [round(res["dur"][k].rank().corr(res["finish"].rank()), 2) for k in tasks.index],
}).sort_values("Чувствительность", ascending=False)
crit

In [ ]:
impact = pd.DataFrame({
    "Риск": risks["name"],
    "Наступил в прогонах, %": (100 * res["fired"].mean()).round(1),
    "Сдвиг финиша, дн": [round(res["finish"][res["fired"][r]].mean() - res["finish"][~res["fired"][r]].mean(), 2) for r in risks.index],
    "P(в срок) без риска, %": [round(100 * (res["finish"][~res["fired"][r]] <= DEADLINE_DAY).mean(), 1) for r in risks.index],
})
impact.sort_values("Сдвиг финиша, дн", ascending=False)

## 6. Дерево решений для риска R2 (пешеходная сеть)

Риск: сеть OSM может оказаться непригодной для расчёта изохрон. Варианты: считать сразу по прямой (дёшево, но при провале переделка) или сначала поставить спайк на 4 часа и проверить сеть на контрольных маршрутах.

In [ ]:
OPTIONS = [
    # вариант,                                  затраты сейчас, ч; исходы: (вероятность, часы, дни задержки)
    ("A. Считать сразу по прямой, без проверки", 0, [(0.40, 12, 3), (0.60, 0, 0)]),
    ("B. Спайк 4 ч: проверить сеть до расчёта",  4, [(0.10, 12, 3), (0.90, 0, 0)]),
]
rows = []
for name, cost, outcomes in OPTIONS:
    assert abs(sum(p for p, _, _ in outcomes) - 1) < 1e-9, f"{name}: вероятности не дают в сумме 1"
    rows.append({
        "Вариант": name,
        "Затраты сейчас, ч": cost,
        "Ожидаемые часы (EMV), ч": round(cost + sum(p * h for p, h, _ in outcomes), 1),
        "Ожидаемая задержка, дн": round(sum(p * d for p, _, d in outcomes), 2),
        "Худший случай, ч": cost + max(h for _, h, _ in outcomes),
    })
pd.DataFrame(rows).set_index("Вариант")

## 7. Что перенести в отчёт

В `docs/risk-register.md`, раздел «Количественный анализ», переносятся:

- вероятность уложиться в срок и в буфер по трём сценариям, срок P50/P85;
- вероятность уложиться в план по часам без резерва и с резервом;
- индекс критичности и чувствительность работ;
- влияние рисков на сдвиг финиша;
- сравнение EMV по дереву решений и вывод для спонсора.

График сохраняется в `docs/img/monte_carlo.png` и вставляется в реестр рисков.

In [ ]:
try:
    from google.colab import files
    files.download("../docs/img/monte_carlo.png")
except ImportError:
    print("Локальный запуск: файл docs/img/monte_carlo.png сохранён в репозитории")